# QBronze 1.3 — Operaciones con qubits reales y múltiples qubits

Este cuaderno desarrolla ejercicios sobre operadores cuánticos reales, rotaciones, reflexiones, producto tensorial, CNOT y orden de qubits.

Usaremos dos convenciones de manera explícita:

1. **Orden lógico matemático:** $|\text{primer qubit},\text{segundo qubit}\rangle$, por ejemplo $|10\rangle$.
2. **Orden de impresión tipo Qiskit:** al medir, la cadena suele imprimirse como $c[n-1]\cdots c[1]c[0]$. Por eso conviene leer con cuidado qué qubit fue medido en qué bit clásico.




In [ ]:
import numpy as np
from math import sqrt, pi, cos, sin, log2, floor
from collections import Counter

np.set_printoptions(precision=6, suppress=True)

ket0 = np.array([1.0, 0.0])
ket1 = np.array([0.0, 1.0])

I = np.eye(2)
X = np.array([[0.0, 1.0], [1.0, 0.0]])
Z = np.array([[1.0, 0.0], [0.0, -1.0]])
H = (1 / sqrt(2)) * np.array([[1.0, 1.0], [1.0, -1.0]])

def tensor(*vectors_or_matrices):
    """Producto tensorial en el orden escrito: A ⊗ B ⊗ C."""
    result = np.array([1.0])
    for item in vectors_or_matrices:
        result = np.kron(result, item)
    return result

def norm2(state):
    """Norma al cuadrado: suma |amplitud|^2."""
    return float(np.vdot(state, state).real)

def probs(state):
    """Probabilidades de medición en la base computacional."""
    return np.abs(state) ** 2

def basis_state(bitstring):
    """Vector base |bitstring⟩ en orden lógico de izquierda a derecha."""
    n = len(bitstring)
    index = int(bitstring, 2)
    v = np.zeros(2**n)
    v[index] = 1.0
    return v

def bitstrings(n):
    return [format(i, f"0{n}b") for i in range(2**n)]

def show_state(state, n=None, tol=1e-10):
    """Representación textual compacta de un vector de estado."""
    if n is None:
        n = int(round(log2(len(state))))
    terms = []
    for amp, bits in zip(state, bitstrings(n)):
        if abs(amp) > tol:
            terms.append(f"{amp:+.6g}|{bits}⟩")
    return " ".join(terms) if terms else "0"

def sample_counts(state, shots=1024, seed=7, qiskit_order=False):
    """Muestreo de una distribución de medición. Si qiskit_order=True, invierte las etiquetas."""
    rng = np.random.default_rng(seed)
    p = probs(state)
    n = int(round(log2(len(state))))
    outcomes = rng.choice(2**n, p=p/p.sum(), size=shots)
    labels = []
    for i in outcomes:
        bits = format(i, f"0{n}b")
        labels.append(bits[::-1] if qiskit_order else bits)
    return dict(sorted(Counter(labels).items()))

def is_valid_quantum_state(v, tol=1e-9):
    return abs(norm2(np.asarray(v, dtype=float)) - 1.0) < tol


def R(theta):
    """Rotación plana usual por ángulo theta sobre el vector de amplitudes reales."""
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])

def RY_qiskit(theta):
    """Matriz RY(theta) con la convención estándar de circuitos: rota el vector real por theta/2."""
    return np.array([[np.cos(theta/2), -np.sin(theta/2)],
                     [np.sin(theta/2),  np.cos(theta/2)]])

def cnot_matrix(control=0, target=1, n=2):
    """Matriz CNOT en orden lógico de bits de izquierda a derecha."""
    size = 2**n
    M = np.zeros((size, size))
    for i, bits in enumerate(bitstrings(n)):
        b = list(bits)
        if b[control] == '1':
            b[target] = '0' if b[target] == '1' else '1'
        j = int(''.join(b), 2)
        M[j, i] = 1.0
    return M

## 1. Rotar un qubit real 120 grados

En circuitos cuánticos se usa la compuerta $RY(\theta)$, cuya matriz es

$$
RY(\theta)=
\begin{pmatrix}
\cos(\theta/2)&-\sin(\theta/2)\\
\sin(\theta/2)&\cos(\theta/2)
\end{pmatrix}.
$$

Si partimos de $|0\rangle=(1,0)^T$, entonces

$$
RY(\theta)|0\rangle=\cos(\theta/2)|0\rangle+\sin(\theta/2)|1\rangle.
$$

Si queremos que el vector real forme un ángulo geométrico de

$$
120^\circ=\frac{2\pi}{3},
$$

entonces necesitamos

$$
\frac{\theta}{2}=\frac{2\pi}{3}
\quad\Longrightarrow\quad
\theta=\frac{4\pi}{3}.
$$

La instrucción correspondiente sería conceptualmente:

```python
qc.ry(2*2*pi/3, q[0])
```

In [ ]:
angle_geometrico = 2*pi/3
parametro_RY = 2 * angle_geometrico

state = RY_qiskit(parametro_RY) @ ket0
print("ángulo geométrico deseado:", angle_geometrico)
print("parámetro de RY:", parametro_RY)
print("estado obtenido:", show_state(state, n=1))
print("vector:", state)
print("esperado: [cos(2π/3), sin(2π/3)] =", [cos(2*pi/3), sin(2*pi/3)])

## 2. La compuerta $Z$ sobre $|0\rangle$

La compuerta $Z$ es

$$
Z=\begin{pmatrix}
1&0\\
0&-1
\end{pmatrix}.
$$

Aplicada sobre $|0\rangle$:

$$
Z|0\rangle=
\begin{pmatrix}
1&0\\
0&-1
\end{pmatrix}
\begin{pmatrix}1\\0\end{pmatrix}
=
\begin{pmatrix}1\\0\end{pmatrix}
=|0\rangle.
$$

Aplicada sobre $|1\rangle$:

$$
Z|1\rangle=-|1\rangle.
$$

La compuerta $Z$ no cambia $|0\rangle$, pero cambia el signo de $|1\rangle$.

In [ ]:
print("Z|0> =", show_state(Z @ ket0, n=1))
print("Z|1> =", show_state(Z @ ket1, n=1))

## 3. Evaluar $HZH|0\rangle$

Calculamos paso a paso, de derecha a izquierda:

$$
HZH|0\rangle = H\bigl(Z(H|0\rangle)\bigr).
$$

Primero:

$$
H|0\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}}.
$$

Luego:

$$
Z\left(\frac{|0\rangle+|1\rangle}{\sqrt{2}}\right)
=\frac{Z|0\rangle+Z|1\rangle}{\sqrt{2}}
=\frac{|0\rangle-|1\rangle}{\sqrt{2}}
=|-\rangle.
$$

Finalmente:

$$
H|-\rangle=|1\rangle.
$$

Entonces:

$$
HZH|0\rangle=|1\rangle.
$$

De hecho,

$$
HZH=X.
$$

In [ ]:
HZH = H @ Z @ H
print("HZH =")
print(HZH)
print("X =")
print(X)
print("HZH|0> =", show_state(HZH @ ket0, n=1))
print("HZH|1> =", show_state(HZH @ ket1, n=1))

## 4. Reflexiones, rotaciones y afirmaciones verdaderas

En el plano real:

- Una rotación pura tiene determinante $+1$.
- Una reflexión tiene determinante $-1$.
- Si $F$ es una reflexión, entonces aplicar la misma reflexión dos veces devuelve el estado original:

$$
F^2=I.
$$

La matriz de Hadamard tiene determinante $-1$, por lo que no es una rotación pura en el plano real: es una reflexión. Además,

$$
H^2=I.
$$

También:

$$
|0\rangle=(1,0),\qquad |1\rangle=(0,1).
$$

Su producto punto es 0, por lo que el ángulo entre ellos es $90^\circ$.

In [ ]:
print("det(H) =", np.linalg.det(H))
print("H^2 =")
print(H @ H)
print("producto punto <0|1> =", np.dot(ket0, ket1))

# Ejemplo de rotación por 30 grados: su determinante es +1.
rot = R(pi/6)
print("det(R(pi/6)) =", np.linalg.det(rot))
print("R(pi/6)^2 no es identidad, salvo ángulos especiales:")
print(rot @ rot)

## 5. Dimensión de un sistema con $n$ qubits

Un qubit tiene 2 estados base:

$$
|0\rangle,\ |1\rangle.
$$

Dos qubits tienen 4 estados base:

$$
|00\rangle,\ |01\rangle,\ |10\rangle,\ |11\rangle.
$$

En general, $n$ qubits tienen

$$
2^n
$$

estados base. Por tanto, el vector de estado tiene dimensión $2^n$.

Para $n=5$:

$$
2^5=32.
$$

In [ ]:
for n in range(1, 8):
    print(f"{n} qubits -> dimensión {2**n}")

## 6. CNOT sobre una superposición

Usamos el orden lógico

$$
|\text{primer qubit},\text{segundo qubit}\rangle.
$$

La compuerta CNOT con primer qubit como control y segundo como objetivo cumple:

$$
|00\rangle\mapsto |00\rangle,
\quad
|01\rangle\mapsto |01\rangle,
\quad
|10\rangle\mapsto |11\rangle,
\quad
|11\rangle\mapsto |10\rangle.
$$

Si el estado inicial es

$$
\frac{|01\rangle+|11\rangle}{\sqrt{2}},
$$

aplicamos CNOT término por término:

$$
|01\rangle\mapsto |01\rangle,
\qquad
|11\rangle\mapsto |10\rangle.
$$

Por tanto,

$$
CNOT\left(\frac{|01\rangle+|11\rangle}{\sqrt{2}}\right)
=
\frac{|01\rangle+|10\rangle}{\sqrt{2}}.
$$

In [ ]:
CNOT_12 = cnot_matrix(control=0, target=1, n=2)
state = (basis_state("01") + basis_state("11")) / sqrt(2)
result = CNOT_12 @ state
print("Estado inicial:", show_state(state, n=2))
print("Estado final:", show_state(result, n=2))

## 7. Crear $(|00\rangle+|01\rangle)/\sqrt{2}$ con un Hadamard

Con el orden de impresión usado por muchos entornos de circuitos, aplicar $H$ a `q[0]` desde $|00\rangle$ produce una superposición en el bit menos significativo:

$$
|00\rangle \xrightarrow{H\text{ en }q[0]} \frac{|00\rangle+|01\rangle}{\sqrt{2}}.
$$

La instrucción conceptual es:

```python
qc.h(0)
```

La interpretación correcta depende del orden de lectura de los bits.

In [ ]:
# En orden lógico |q1 q0>, aplicar H sobre q0 equivale a I ⊗ H.
state = basis_state("00")
result = tensor(I, H) @ state
print("Estado final:", show_state(result, n=2))

## 8. Cambiar una correlación perfecta a anticorrelación

El estado de Bell

$$
|\Phi^+\rangle=\frac{|00\rangle+|11\rangle}{\sqrt{2}}
$$

está correlacionado: las mediciones posibles son $00$ y $11$.

Si aplicamos $X$ al primer qubit:

$$
X\otimes I:
\quad
|00\rangle\mapsto |10\rangle,
\quad
|11\rangle\mapsto |01\rangle.
$$

Entonces

$$
(X\otimes I)|\Phi^+\rangle
=
\frac{|10\rangle+|01\rangle}{\sqrt{2}}.
$$

Ahora las mediciones posibles son $10$ y $01$, es decir, resultados anticorrelacionados.

In [ ]:
phi_plus = (basis_state("00") + basis_state("11")) / sqrt(2)
anti = tensor(X, I) @ phi_plus
print("Bell correlacionado:", show_state(phi_plus, n=2))
print("Después de X en el primer qubit:", show_state(anti, n=2))
print("Muestreo:", sample_counts(anti, shots=1000, seed=4))

## 9. Aplicar $H$ sólo al segundo qubit de tres qubits

Si partimos de

$$
|000\rangle
$$

y aplicamos $H$ únicamente al segundo qubit, los otros qubits se comportan como si se les aplicara identidad:

$$
I\otimes H\otimes I.
$$

Entonces:

$$
|000\rangle
\xrightarrow{I\otimes H\otimes I}
|0\rangle\otimes \frac{|0\rangle+|1\rangle}{\sqrt{2}}\otimes |0\rangle
=
\frac{|000\rangle+|010\rangle}{\sqrt{2}}.
$$

Sólo cambia el qubit sobre el que se aplica la compuerta.

In [ ]:
state = basis_state("000")
result = tensor(I, H, I) @ state
print("Resultado:", show_state(result, n=3))
print("Probabilidades no nulas:")
for bits, p in zip(bitstrings(3), probs(result)):
    if p > 1e-12:
        print(bits, p)

## 10. Compuertas controladas por 0

Una compuerta controlada usualmente se activa cuando el control está en $|1\rangle$.
Si se desea activar una compuerta cuando el control está en $|0\rangle$, se usa el patrón:

$$
X \;\longrightarrow\; \text{control sobre }1 \;\longrightarrow\; X.
$$

La primera $X$ cambia $|0\rangle$ a $|1\rangle$. La compuerta controlada se activa. La segunda $X$ restaura el valor original del control.

Esto es útil para entender por qué sí es posible aplicar una NOT a un objetivo dependiendo de que otro qubit esté en $|0\rangle$.

In [ ]:
def controlled_on_zero_not_state(control_bit, target_bit):
    """Demuestra el efecto clásico de un NOT controlado por control=0."""
    if control_bit == 0:
        target_bit = 1 - target_bit
    return control_bit, target_bit

for c in [0, 1]:
    for t in [0, 1]:
        print(f"entrada control={c}, target={t} -> salida {controlled_on_zero_not_state(c, t)}")

## 11. Resumen operativo del módulo

Resultados clave:

$$
RY(\theta)|0\rangle=\cos(\theta/2)|0\rangle+\sin(\theta/2)|1\rangle.
$$

$$
Z|0\rangle=|0\rangle,
\qquad
Z|1\rangle=-|1\rangle.
$$

$$
HZH=X.
$$

$$
\dim(\text{sistema de }n\text{ qubits})=2^n.
$$

$$
CNOT|a,b\rangle=|a,b\oplus a\rangle.
$$